# Failure-model feature pipeline: pandas vs cudf.pandas (Google Colab, T4)

Part of [services-ops-intelligence](https://github.com/arnavchawla26/services-ops-intelligence).
This notebook builds the same per-drive features that `sois.failure_model` uses
(current SMART 5/187/188/197/198 raw and normalized, 9/194/199/241/242 raw,
7-day delta and 7-day max of the reallocated/pending/uncorrectable counts,
days seen) on N days of the real Backblaze 2026 Q2 daily CSVs, first with
plain pandas, then with NVIDIA cuDF through `cudf.pandas`, and prints a
timings table.

## Exact steps

1. Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU** -> Save.
2. Optionally change `N_DAYS` in the next cell (8 is the minimum for the
   7-day features; 30 is the default; 91 is the whole quarter, about 11.7 GB
   of CSV, which fits Colab's disk but takes a while to unzip).
3. Runtime -> Run all. The download is 2.1 GB from Backblaze's public bucket;
   no account or secret is needed. Expect 5 to 15 minutes end to end at N_DAYS=30.
4. Copy the output of the last cell (the timings table and the environment
   block) into `docs/COLAB.md` in the repository under "Measured timings".

`cudf.pandas` must be active before pandas is imported, so each run happens in
its own process: `python features_bench.py` for pandas and
`python -m cudf.pandas features_bench.py` for the GPU run. Nothing else differs.


In [ ]:
N_DAYS = 30          # 8 minimum, 91 = whole quarter
ZIP_URL = "https://f001.backblazeb2.com/file/Backblaze-Hard-Drive-Data/data_Q2_2026.zip"


In [ ]:
import subprocess, sys, platform, json
print("python", sys.version.split()[0], "|", platform.platform())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU visible: set the runtime to T4 GPU")
import pandas; print("pandas", pandas.__version__)
try:
    import cudf; print("cudf", cudf.__version__)
except Exception as e:
    print("cudf import failed:", e, "(cudf.pandas is preinstalled on Colab GPU runtimes; on CPU runtimes it is absent)")


In [ ]:
%%bash -s "$ZIP_URL" "$N_DAYS"
set -e
mkdir -p /content/bb && cd /content/bb
[ -f data_Q2_2026.zip ] || wget -q --show-progress -c "$1" -O data_Q2_2026.zip
# extract only the first N_DAYS daily files, in date order
unzip -Z1 data_Q2_2026.zip | grep '\.csv$' | sort | head -n "$2" > members.txt
unzip -q -o data_Q2_2026.zip $(cat members.txt) -d .
ls data_Q2_2026 | head -3; echo "..."; ls data_Q2_2026 | wc -l
du -sh data_Q2_2026


In [ ]:
%%writefile /content/features_bench.py
"""Failure-model feature pipeline on N daily Backblaze CSVs, plain pandas.
Run as   python features_bench.py <csv_dir> <n_days> <out.json>           (pandas)
or       python -m cudf.pandas features_bench.py <csv_dir> <n_days> <out.json>   (GPU)
Mirrors sois.failure_model.build_features: for the last day t, per drive the
current SMART 5/187/188/197/198 raw+normalized, 9/194/199/241/242 raw,
capacity, model, the 7-day delta and 7-day max of 5/187/197/198, days seen.
"""
import json, sys, time, glob, os
t_start = time.perf_counter()
import pandas as pd

csv_dir, n_days, out = sys.argv[1], int(sys.argv[2]), sys.argv[3]
files = sorted(glob.glob(os.path.join(csv_dir, "*.csv")))[:n_days]
BB = (5, 187, 188, 197, 198); EXTRA = (9, 194, 199, 241, 242); DELTA = (5, 187, 197, 198)
cols = ["date", "serial_number", "model", "capacity_bytes", "failure"]
cols += [f"smart_{n}_{k}" for n in BB for k in ("raw", "normalized")] + [f"smart_{n}_raw" for n in EXTRA]

t0 = time.perf_counter()
frames = [pd.read_csv(f, usecols=lambda c: c in cols) for f in files]
df = pd.concat(frames, ignore_index=True)
df["date"] = pd.to_datetime(df["date"])
t_read = time.perf_counter() - t0

t0 = time.perf_counter()
t = df["date"].max(); lag_day = df["date"].min()
cur = df[df["date"] == t].set_index("serial_number")
lag = df[df["date"] == lag_day].set_index("serial_number")[[f"smart_{n}_raw" for n in DELTA]]
lag.columns = [f"smart_{n}_lag" for n in DELTA]
agg = df.groupby("serial_number").agg(**{f"smart_{n}_max7": (f"smart_{n}_raw", "max") for n in DELTA}, days_seen_7=("date", "count"))
feat = cur.join(lag, how="left").join(agg, how="left")
for n in DELTA:
    feat[f"smart_{n}_delta7"] = feat[f"smart_{n}_raw"] - feat[f"smart_{n}_lag"]
feat["heuristic_flag"] = sum((feat[f"smart_{n}_raw"].fillna(0) > 0).astype(int) for n in BB) > 0
t_feat = time.perf_counter() - t0

res = {"mode": "cudf.pandas" if "cudf" in sys.modules or any("cudf" in m for m in sys.modules) else "pandas",
       "days": len(files), "rows_read": int(len(df)), "drives_on_last_day": int(len(feat)),
       "flagged_on_last_day": int(feat["heuristic_flag"].sum()),
       "read_seconds": round(t_read, 2), "feature_seconds": round(t_feat, 2),
       "total_seconds": round(time.perf_counter() - t_start, 2), "pandas_version": pd.__version__}
try:
    import cudf; res["cudf_version"] = cudf.__version__
except Exception:
    pass
json.dump(res, open(out, "w"), indent=2); print(json.dumps(res, indent=2))


In [ ]:
import json, subprocess, time
runs = {}
for mode, cmd in {"pandas": ["python"], "cudf.pandas": ["python", "-m", "cudf.pandas"]}.items():
    out = f"/content/bench_{mode.replace('.', '_')}.json"
    t0 = time.perf_counter()
    r = subprocess.run(cmd + ["/content/features_bench.py", "/content/bb/data_Q2_2026", str(N_DAYS), out],
                       capture_output=True, text=True)
    wall = time.perf_counter() - t0
    if r.returncode != 0:
        print(mode, "FAILED\n", r.stderr[-3000:])
        continue
    runs[mode] = json.load(open(out)); runs[mode]["process_wall_seconds"] = round(wall, 2)
    print(mode, "ok in", round(wall, 1), "s")


In [ ]:
import pandas as pd
if {"pandas", "cudf.pandas"} <= set(runs):
    p, c = runs["pandas"], runs["cudf.pandas"]
    rows = []
    for k, label in (("read_seconds", "read N CSVs (needed columns)"), ("feature_seconds", "features: lag, 7-day max, delta, join"),
                     ("total_seconds", "script total (incl. import)"), ("process_wall_seconds", "process wall clock")):
        rows.append({"step": label, "pandas_s": p[k], "cudf_pandas_s": c[k], "speedup": round(p[k] / max(c[k], 1e-9), 2)})
    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    print()
    print("days:", p["days"], "| rows read:", p["rows_read"], "| drives on last day:", p["drives_on_last_day"],
          "| heuristic-flagged:", p["flagged_on_last_day"], "(cudf run:", c["drives_on_last_day"], c["flagged_on_last_day"], ")")
    print("pandas", p["pandas_version"], "| cudf", c.get("cudf_version", "n/a"))
    print(subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    print()
    print("Markdown for docs/COLAB.md:")
    print(table.to_markdown(index=False))
else:
    print("one of the runs failed; see the cell above")
